In [1]:
#import necessary libraries

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score , balanced_accuracy_score , classification_report

In [2]:
#Load data

df = pd.read_csv(r"c:\Users\alvin\OneDrive\Desktop\student_dataset_10000_rows.csv")
print(df.shape)

(10000, 8)


In [3]:
#Check for empty spaces

print(df.isnull().sum())

study_hours              0
attendance               0
sleep_hours              0
internet_usage           0
assignments_completed    0
previous_score           0
exam_score               0
placement_status         0
dtype: int64


In [6]:
print(df.columns)

Index(['study_hours', 'attendance', 'sleep_hours', 'internet_usage',
       'assignments_completed', 'previous_score', 'exam_score',
       'placement_status', 'placement_encoded'],
      dtype='str')


In [5]:
#Encode text columns

le = LabelEncoder()
df['placement_encoded'] = le.fit_transform(df['placement_status'])

In [7]:
#Correlation of inputs to output

print(df.drop(columns=['placement_status']).corr()['placement_encoded'].sort_values(ascending=False))

placement_encoded        1.000000
exam_score               0.792392
study_hours              0.400934
assignments_completed    0.280138
previous_score           0.232693
attendance               0.176738
sleep_hours              0.110578
internet_usage          -0.110233
Name: placement_encoded, dtype: float64


In [8]:
#Identify inputs and output

X = df[['study_hours' , 'assignments_completed' , 'previous_score']]
y = df['placement_encoded']
print("\nInput X: ")
print(X)
print("\nOutput y: ")
print(y)


Input X: 
      study_hours  assignments_completed  previous_score
0               7                     10              62
1               4                      8              56
2              11                     10              45
3               8                      4              55
4               5                      8              40
...           ...                    ...             ...
9995            2                      8              88
9996            7                      4              87
9997           10                     10              37
9998           10                      8              52
9999            2                     16              52

[10000 rows x 3 columns]

Output y: 
0       1
1       1
2       1
3       1
4       1
       ..
9995    0
9996    1
9997    1
9998    1
9999    1
Name: placement_encoded, Length: 10000, dtype: int64


In [9]:
#Check shape for confirmation

print(X.shape)
print(y.shape)

(10000, 3)
(10000,)


In [10]:
#Split into training and testing groups

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

Training rows: 8000
Testing rows: 2000


In [29]:
#Fit model with data

model = RandomForestClassifier(n_estimators=200 , max_depth=10 , min_samples_split=3 , min_samples_leaf=1 ,class_weight='balanced' , n_jobs=-1)
model.fit(X_train , y_train)
print("Model trained successfully!")

Model trained successfully!


In [30]:
#Test model
predictions = model.predict(X_test)
print(f"Predictions: {predictions}")
print(f"Actual: {y_test.values}")

Predictions: [1 1 1 ... 1 1 0]
Actual: [1 1 1 ... 1 1 0]


In [31]:
#Evaluate model

accuracy = accuracy_score(y_test , predictions)
balanced = balanced_accuracy_score(y_test , predictions)
report = classification_report(y_test , predictions)
print(f"Accuracy: {accuracy:.2f}")
print(f"Balanced: {balanced:.2f}")
print(report)

Accuracy: 0.85
Balanced: 0.81
              precision    recall  f1-score   support

           0       0.55      0.76      0.64       345
           1       0.94      0.87      0.91      1655

    accuracy                           0.85      2000
   macro avg       0.75      0.81      0.77      2000
weighted avg       0.88      0.85      0.86      2000



In [32]:
#Check feature importance

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance)

                 Feature  Importance
0            study_hours    0.439025
1  assignments_completed    0.280976
2         previous_score    0.279999


In [35]:
#Cross validate model

from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model, X, y, 
    cv=10,
    scoring='accuracy'
)

print(f"Scores: {scores}")
print(f"Average: {scores.mean():.2f}")
print(f"Std Dev: {scores.std():.2f}")

Scores: [0.838 0.836 0.837 0.857 0.859 0.831 0.845 0.84  0.844 0.829]
Average: 0.84
Std Dev: 0.01
